# NuminaMath SFT training on processed data

This notebook trains a LoRA adapter on the conversational dataset produced by `NumiNaMath_Preprocess_for_SFT.ipynb`, uses the default SFTTrainer evaluation path, and finishes with a couple of generation examples in eval mode.

## 1. Environment and dataset path

Configure HuggingFace cache variables before importing libraries, then locate the processed dataset saved by the preprocessing notebook.

In [ ]:
import os
from pathlib import Path


def resolve_project_root() -> Path:
    """Return the repository root whether the notebook runs from the repo root or notebooks/."""
    cwd = Path.cwd().resolve()
    if (cwd / "pyproject.toml").exists():
        return cwd
    if (cwd.parent / "pyproject.toml").exists():
        return cwd.parent
    return cwd


PROJECT_ROOT = resolve_project_root()
CACHE_ROOT = PROJECT_ROOT / ".cache" / "huggingface"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(CACHE_ROOT))
os.environ.setdefault("HF_DATASETS_CACHE", str(CACHE_ROOT / "datasets"))
os.environ.setdefault("HF_HUB_CACHE", str(CACHE_ROOT / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(CACHE_ROOT / "transformers"))

DATASETS_DIR = PROJECT_ROOT / "datasets"
PROCESSED_DATASET_DIR = DATASETS_DIR / "processed" / "numinamath_sft" / "train_990_seed_42"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CACHE_ROOT:   {CACHE_ROOT}")
print(f"PROCESSED_DATASET_DIR: {PROCESSED_DATASET_DIR}")

## 2. Load the processed conversational dataset

Load the saved dataset from disk and verify that each example has the prompt/completion conversation structure expected by `SFTTrainer`.

In [ ]:
from typing import cast

from datasets import Dataset, load_from_disk


# Quick smoke-run flags: set USE_SUBSET=True to select small subsets for fast testing
USE_SUBSET = True
TRAIN_SUBSET_SIZE = 2000
TEST_SUBSET_SIZE = 50

if not PROCESSED_DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at {PROCESSED_DATASET_DIR}. Run NumiNaMath_Preprocess_for_SFT.ipynb first."
    )

processed_dataset = load_from_disk(str(PROCESSED_DATASET_DIR))
train_dataset = cast(Dataset, processed_dataset["train"])
test_dataset = cast(Dataset, processed_dataset["test"])

# Optionally use small subsets for quick validation of the pipeline
if USE_SUBSET:
    n_train = min(TRAIN_SUBSET_SIZE, len(train_dataset))
    n_test = min(TEST_SUBSET_SIZE, len(test_dataset))
    train_dataset = train_dataset.select(range(n_train))
    test_dataset = test_dataset.select(range(n_test))
    print(f"Using subset: train={n_train}, test={n_test}")

required_keys = {"prompt", "completion"}
first_train_sample = train_dataset[0]
first_test_sample = test_dataset[0]

assert required_keys.issubset(first_train_sample.keys()), f"Missing keys in train sample: {first_train_sample.keys()}"
assert required_keys.issubset(first_test_sample.keys()), f"Missing keys in test sample: {first_test_sample.keys()}"
assert isinstance(first_train_sample["prompt"], list) and first_train_sample["prompt"], "prompt must be a non-empty conversational list"
assert isinstance(first_train_sample["completion"], list) and first_train_sample["completion"], "completion must be a non-empty conversational list"
assert first_train_sample["prompt"][0]["role"] == "user"
assert first_train_sample["completion"][0]["role"] == "assistant"

def validate_conversational_tags(dataset, split_name):
    missing_tag_indices = []
    for sample_index, sample in enumerate(dataset):
        prompt = sample["prompt"]
        completion = sample["completion"]
        assert isinstance(prompt, list) and prompt, f"{split_name} sample {sample_index} has an invalid prompt"
        assert isinstance(completion, list) and completion, f"{split_name} sample {sample_index} has an invalid completion"
        assert prompt[0]["role"] == "user", f"{split_name} sample {sample_index} prompt must start with user"
        assert completion[0]["role"] == "assistant", f"{split_name} sample {sample_index} completion must start with assistant"
        completion_text = completion[0].get("content", "")
        if ("<think>" not in completion_text or "</think>" not in completion_text or "<answer>" not in completion_text or "</answer>" not in completion_text):
            missing_tag_indices.append(sample_index)
    assert not missing_tag_indices, f"{split_name} samples missing required tags at indices: {missing_tag_indices}"

validate_conversational_tags(train_dataset, "train")
validate_conversational_tags(test_dataset, "test")

print(processed_dataset)
print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")
print("Verified conversational prompt/completion format.")

In [ ]:
import torch
from typing import cast

from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer
import numpy as np


MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_DIR = PROJECT_ROOT / "models" / "qwen3-0.6B-SFT"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not any(MODEL_DIR.iterdir()):
    print(f"{MODEL_DIR} is empty -> downloading model/tokenizer from Hub and saving to disk...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    tokenizer.save_pretrained(str(MODEL_DIR))
    model.save_pretrained(str(MODEL_DIR))
else:
    print(f"{MODEL_DIR} is not empty -> loading model/tokenizer from disk...")
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def count_params(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return trainable, total


base_model = model
full_trainable, full_total = count_params(base_model)
print(f"Model: {MODEL_ID}, full FT trainable: {full_trainable:,}")

RANK = 4
peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    inference_mode=False,
    r=RANK,
    lora_alpha=2 * RANK,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

lora_model = cast(PeftModel, get_peft_model(base_model, peft_config))

lora_trainable, lora_total = count_params(lora_model)
reduction_vs_full_ft = 100 * (1 - (lora_trainable / full_trainable))
trainable_pct_of_total = 100 * (lora_trainable / lora_total)

model_for_eval = lora_model
trainer = None

print(f"LoRA trainable:    {lora_trainable:,}")
print(f"Reduction:         {reduction_vs_full_ft:.2f}%")
print(f"LoRA trainable %:  {trainable_pct_of_total:.4f}%")

## 3. Train the adapter

Train the model

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "sft-numinamath-processed"
LOGGING_DIR = PROJECT_ROOT / "logs"

# Smoke-run helper: set SMOKE_RUN=True to override epochs for quick tests
SMOKE_RUN = True
NUM_EPOCHS = 1 if SMOKE_RUN else 3

sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=NUM_EPOCHS,
    logging_strategy="steps",
    logging_dir=str(LOGGING_DIR),
    logging_steps=10,
    gradient_checkpointing=True,
    bf16=True,
    eval_strategy="steps",
    eval_steps=10,
    save_steps=100,
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    eval_accumulation_steps=8,
    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=cast(PeftModel, lora_model),
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
)

print(trainer)
print("Trainer train dataset sample keys:", train_dataset[0].keys())
print("Trainer eval dataset sample keys:", test_dataset[0].keys())

In [ ]:
trainer.train()

model_for_eval = trainer.model
model_for_eval.eval()
print("Training complete. Model switched to eval mode.")

# 4. Sample Evaluation 

In [ ]:
def extract_assistant_completion(completion):
    """Extract the assistant content from a conversational completion record."""
    if isinstance(completion, list) and completion:
        return completion[0].get("content", "")
    return str(completion)


def parse_completion(completion_text):
    """Parse <think> and <answer> tags from generated text."""
    reasoning = ""
    final_answer = "no answer tags"

    if "<think>" in completion_text and "</think>" in completion_text:
        start_idx = completion_text.find("<think>") + len("<think>")
        end_idx = completion_text.find("</think>")
        reasoning = completion_text[start_idx:end_idx].strip()

    if "<answer>" in completion_text and "</answer>" in completion_text:
        answer_start = completion_text.find("<answer>") + len("<answer>")
        answer_end = completion_text.find("</answer>")
        final_answer = completion_text[answer_start:answer_end].strip() or "no answer tags"

    return reasoning, final_answer


def extract_original_problem(sample):
    prompt_messages = sample["prompt"]
    if isinstance(prompt_messages, list) and prompt_messages:
        return prompt_messages[0].get("content", "")
    return str(prompt_messages)


N_EVAL_EXAMPLES = 5
EVAL_SEED = 123
MAX_NEW_TOKENS = 2048
rng = np.random.default_rng(EVAL_SEED)
eval_indices = rng.choice(len(test_dataset), size=min(N_EVAL_EXAMPLES, len(test_dataset)), replace=False).tolist()

for i, idx in enumerate(eval_indices):
    sample = test_dataset[idx]
    original_problem = extract_original_problem(sample)
    prompt_messages = sample["prompt"]
    ground_truth_completion = extract_assistant_completion(sample["completion"])
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model_for_eval.device)

    with torch.no_grad():
        generated = model_for_eval.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = generated[0][prompt_length:]
    predicted_completion = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    gt_reasoning, gt_answer = parse_completion(ground_truth_completion)
    pred_reasoning, pred_answer = parse_completion(predicted_completion)

    print("=" * 80)
    print(f"Example {i+1} / {N_EVAL_EXAMPLES}")
    print("Original problem:")
    print(original_problem)
    print()
    # print("Ground truth chain of thought:")
    # print(gt_reasoning or "[empty]")
    # print()
    # print("Predicted chain of thought:")
    # print(pred_reasoning or "[empty]")
    # print()
    print("Ground truth answer:")
    print(gt_answer or "[empty]")
    print()
    print("Predicted answer:")
    print(pred_answer or "[empty]")

# 5. Save model

In [ ]:
SAVE_DIR = OUTPUT_DIR / "final_adapter"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))

print(f"Saved LoRA adapter and tokenizer to {SAVE_DIR}")

## 6. Run evaluation on the whole test set

In [ ]:
from tqdm.auto import tqdm
from bert_score import BERTScorer
from typing import cast

full_test_dataset = cast(Dataset, processed_dataset["test"])

# Use a subset for metrics to keep compute reasonable
TEST_METRIC_SUBSET_SIZE = min(TEST_SUBSET_SIZE, len(full_test_dataset))
metric_dataset = full_test_dataset.select(range(TEST_METRIC_SUBSET_SIZE))


def extract_assistant_completion(completion):
    if isinstance(completion, list) and completion:
        return completion[0].get("content", "")
    return str(completion)


def parse_completion(completion_text):
    reasoning = ""
    final_answer = "no answer tags"

    if "<think>" in completion_text and "</think>" in completion_text:
        start_idx = completion_text.find("<think>") + len("<think>")
        end_idx = completion_text.find("</think>")
        reasoning = completion_text[start_idx:end_idx].strip()

    if "<answer>" in completion_text and "</answer>" in completion_text:
        answer_start = completion_text.find("<answer>") + len("<answer>")
        answer_end = completion_text.find("</answer>")
        final_answer = completion_text[answer_start:answer_end].strip() or "no answer tags"

    return reasoning, final_answer

predicted_answers = []
ground_truth_answers = []
example_records = []

for index, sample in enumerate(tqdm(metric_dataset, total=len(metric_dataset), desc="Predicting test answers")):
    prompt_messages = sample["prompt"]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model_for_eval.device)

    with torch.no_grad():
        generated = model_for_eval.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = generated[0][prompt_length:]
    predicted_completion = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    _, predicted_answer = parse_completion(predicted_completion)
    _, ground_truth_answer = parse_completion(extract_assistant_completion(sample["completion"]))

    predicted_answers.append(predicted_answer)
    ground_truth_answers.append(ground_truth_answer)
    example_records.append(
        {
            "index": index,
            "problem": prompt_messages[0].get("content", "") if prompt_messages else "",
            "predicted_answer": predicted_answer,
            "ground_truth_answer": ground_truth_answer,
        }
    )

scorer = BERTScorer(
    lang="en",
    model_type="distilbert-base-uncased",
    rescale_with_baseline=True,
    batch_size=16,
    device=str(model_for_eval.device),
)
precision, recall, f1 = scorer.score(predicted_answers, ground_truth_answers)
bert_f1_scores = f1.detach().cpu().tolist()
mean_bert_f1 = sum(bert_f1_scores) / len(bert_f1_scores) if bert_f1_scores else 0.0
worst_examples = sorted(zip(bert_f1_scores, example_records), key=lambda item: item[0])[:3]

print(f"Metric subset size: {len(metric_dataset)}")
print(f"Mean BERTScore F1: {mean_bert_f1:.4f}")
print("Worst 3 BERTScore F1 cases:")
for score, example in worst_examples:
    print("-" * 80)
    print(f"Index: {example['index']}")
    print(f"BERTScore F1: {score:.4f}")
    print("Problem:")
    print(example["problem"])
    print("Ground truth answer:")
    print(example["ground_truth_answer"])
    print("Predicted answer:")
    print(example["predicted_answer"])
